# Part 8: Training Transformers — From Loss to a Trained Model

In notebooks 02–07 we built a GPT and *used* it. Now we look hard at the thing that
actually makes it work: **training**. We'll go from the objective (next-token
prediction as maximum likelihood) all the way to a correct, instrumented training
loop, plus the engineering tricks that make training stable and scalable.

What we cover:

1. What training optimizes — cross-entropy as maximum likelihood
2. Perplexity as an interpretable metric
3. Train/val split and why held-out data matters
4. The AdamW optimizer
5. Learning-rate schedules: warmup + cosine decay
6. Gradient clipping
7. A correct training loop tracking train **and** val loss
8. Overfitting on a tiny dataset
9. The overfit-a-single-batch sanity check
10. Scaling techniques: grad accumulation, mixed precision, checkpointing
11. The bigger picture: pretraining → fine-tuning → instruction-tuning → RLHF

---

In [ ]:
import sys
sys.path.append('..')

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from src.gpt import GPT, create_gpt_small
from src.train import CharTokenizer, TextDataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
print(f"Using device: {device}")

## Step 1: What Training Optimizes

A language model defines a probability distribution over the next token given the
context: $p_\theta(x_t \mid x_{<t})$. Training is **maximum likelihood estimation**:
we choose parameters $\theta$ that maximize the probability of the observed text.

The likelihood of a sequence factorizes autoregressively:

$$p_\theta(x_1, \dots, x_T) = \prod_{t=1}^{T} p_\theta(x_t \mid x_{<t})$$

Maximizing the (log-)likelihood is the same as minimizing the **negative log-likelihood**,
which for a categorical distribution is exactly the **cross-entropy loss**:

$$\mathcal{L} = -\frac{1}{T}\sum_{t=1}^{T} \log p_\theta(x_t \mid x_{<t})$$

In code, the model produces `logits` of shape `(B, T, vocab)` and the targets are the
next tokens of shape `(B, T)`. `F.cross_entropy` expects 2D logits `(N, vocab)` and 1D
targets `(N,)`, so we flatten the batch and time dimensions together — this is exactly
what `GPT.forward` does internally.

In [ ]:
# Cross-entropy over the flattened (B*T, vocab) view, by hand.
B, T, V = 2, 5, 11
torch.manual_seed(0)
logits = torch.randn(B, T, V)
targets = torch.randint(0, V, (B, T))

# Flatten time and batch into one axis of independent predictions.
loss_flat = F.cross_entropy(logits.view(-1, V), targets.view(-1))

# Equivalent: average per-token NLL computed manually.
logp = F.log_softmax(logits, dim=-1)
nll = -logp.gather(-1, targets.unsqueeze(-1)).squeeze(-1)  # (B, T)
loss_manual = nll.mean()

print(f"F.cross_entropy : {loss_flat.item():.4f}")
print(f"manual NLL mean : {loss_manual.item():.4f}")
assert torch.allclose(loss_flat, loss_manual, atol=1e-5)

## Step 2: Perplexity

Cross-entropy is in nats and hard to feel. **Perplexity** maps it back to an intuitive
scale:

$$\text{perplexity} = \exp(\mathcal{L})$$

Interpretation: the model is, on average, as confused as if it were uniformly guessing
among `perplexity` equally-likely tokens. So a useful sanity anchor is the
**random baseline**: an untrained model assigns roughly uniform probability over the
vocab, giving loss $\approx \log(\text{vocab\_size})$ and perplexity $\approx$
`vocab_size`. Training should drive perplexity well below that.

In [ ]:
def perplexity(loss):
    return math.exp(loss)

vocab_demo = 65
random_loss = math.log(vocab_demo)
print(f"Random-baseline loss       : {random_loss:.4f}")
print(f"Random-baseline perplexity : {perplexity(random_loss):.2f}  (~= vocab_size {vocab_demo})")
print(f"A trained char model might reach loss ~1.5 -> perplexity {perplexity(1.5):.2f}")

## Step 3: Train/Val Split

We split the corpus into a **training** portion and a held-out **validation** portion.
The tokenizer is fit on the *full* text (so the vocab is identical for both splits),
but the model only ever *learns* from the train split.

Why held-out data matters: training loss only tells you how well the model memorizes
data it has seen. Validation loss estimates **generalization** — performance on text the
model never updated its weights on. The gap between the two is the headline diagnostic
for overfitting.

In [ ]:
data_path = '../data/sample_text.txt'
with open(data_path, 'r', encoding='utf-8') as f:
    text = f.read()
print(f"Total characters: {len(text):,}")

# Tokenizer on the FULL text so vocab is shared across splits.
tokenizer = CharTokenizer(text)
vocab_size = tokenizer.vocab_size
print(f"Vocab size: {vocab_size}")

# 90/10 contiguous split (contiguous keeps val text coherent, not interleaved).
split = int(0.9 * len(text))
train_text, val_text = text[:split], text[split:]
print(f"Train chars: {len(train_text):,}  |  Val chars: {len(val_text):,}")

SEQ_LEN = 64
BATCH_SIZE = 32

train_ds = TextDataset(train_text, tokenizer, SEQ_LEN)
val_ds   = TextDataset(val_text, tokenizer, SEQ_LEN)
print(f"Train samples: {len(train_ds):,}  |  Val samples: {len(val_ds):,}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
print(f"Train batches/epoch: {len(train_loader)}  |  Val batches: {len(val_loader)}")

## Step 4: The Optimizer — AdamW

Transformers are essentially always trained with **AdamW**. Adam keeps per-parameter
running estimates of the gradient mean (`beta1`) and uncentered variance (`beta2`),
giving an adaptive, roughly scale-invariant step size — important because the loss
landscape of a transformer is badly conditioned and a single global learning rate (plain
SGD) struggles.

- `betas=(0.9, 0.95)` is common for transformers (slightly lower beta2 than the 0.999
  default reacts faster to changing gradient statistics on large models).
- `weight_decay` (≈0.1) regularizes by pulling weights toward zero.

**Why AdamW and not Adam?** Plain Adam folds L2 regularization into the gradient, so the
decay term gets scaled by the adaptive per-parameter learning rate — meaning weight decay
interacts with the gradient history in unintended ways. AdamW **decouples** weight decay:
it applies it directly to the weights as a separate step, which is what you actually want.
That decoupling is the entire difference, and it matters in practice.

In [ ]:
def make_model():
    model = create_gpt_small(vocab_size, max_seq_len=SEQ_LEN).to(device)
    return model

model = make_model()
print(f"Model parameters: {model.count_parameters():,}")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-3,            # higher than usual: tiny model + tiny data, train fast
    betas=(0.9, 0.95),
    weight_decay=0.1,
)
print(optimizer)

## Step 5: Learning-Rate Schedule — Warmup + Cosine Decay

A constant learning rate is rarely optimal. The standard transformer schedule is:

1. **Linear warmup** from 0 to the peak LR over the first few hundred steps.
2. **Cosine decay** from the peak down toward (near) zero over the rest of training.

**Why warmup?** Early in training the parameters are random and the adaptive moment
estimates in Adam are still garbage (the variance estimate hasn't accumulated). Taking
full-size steps then can blow up activations through the residual stream and destabilize
LayerNorm statistics. Warmup eases in gently until the optimizer state is trustworthy.

We implement it as a multiplicative factor on the base LR via `LambdaLR`.

In [ ]:
EPOCHS = 10
STEPS_PER_EPOCH = 60          # cap steps/epoch so this trains fast on CPU
steps_per_epoch = min(STEPS_PER_EPOCH, len(train_loader))
total_steps = EPOCHS * steps_per_epoch
warmup_steps = max(1, int(0.1 * total_steps))

def lr_lambda(step):
    if step < warmup_steps:
        return step / warmup_steps                      # linear warmup 0 -> 1
    # cosine decay 1 -> 0 over remaining steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# Plot the schedule (multiplier * base_lr).
base_lr = optimizer.param_groups[0]['lr']
xs = np.arange(total_steps)
ys = np.array([lr_lambda(s) * base_lr for s in xs])

plt.figure(figsize=(7, 3))
plt.plot(xs, ys)
plt.axvline(warmup_steps, color='red', ls='--', alpha=0.6, label=f'warmup end ({warmup_steps})')
plt.xlabel('step'); plt.ylabel('learning rate'); plt.title('Warmup + Cosine Decay')
plt.legend(); plt.tight_layout(); plt.show()
print(f"total_steps={total_steps}, warmup_steps={warmup_steps}, peak_lr={base_lr}")

## Step 6: Gradient Clipping

Transformers can occasionally produce huge gradients (an unlucky batch, a sharp region of
the loss surface), and Adam's adaptivity doesn't fully protect against the resulting
parameter jump. **Gradient clipping** rescales the whole gradient vector so its global
L2 norm never exceeds a threshold (typically `1.0`):

```python
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
```

This caps the step size without changing the gradient *direction*, preventing the
occasional spike from derailing training. It goes **after** `loss.backward()` and
**before** `optimizer.step()`.

In [ ]:
# Demonstrate the rescaling: clip_grad_norm_ returns the PRE-clip total norm.
demo = make_model()
xb, yb = next(iter(train_loader))
xb, yb = xb.to(device), yb.to(device)
_, loss = demo(xb, yb)
loss.backward()

total_norm_before = torch.sqrt(sum((p.grad**2).sum() for p in demo.parameters() if p.grad is not None))
returned = torch.nn.utils.clip_grad_norm_(demo.parameters(), max_norm=1.0)
total_norm_after = torch.sqrt(sum((p.grad**2).sum() for p in demo.parameters() if p.grad is not None))
print(f"grad norm before clip: {total_norm_before.item():.4f}  (returned: {returned.item():.4f})")
print(f"grad norm after  clip: {total_norm_after.item():.4f}  (capped at 1.0)")
del demo

## Step 7: A Correct Training Loop

Now the real thing. Every iteration:

1. move the batch to `device`
2. forward pass to get `loss`
3. `optimizer.zero_grad()` — clear stale grads
4. `loss.backward()` — backprop
5. `clip_grad_norm_` — cap gradient norm
6. `optimizer.step()` — update weights
7. `scheduler.step()` — advance the LR schedule

After each epoch we run an `@torch.no_grad()` evaluation pass over the val set with the
model in `eval()` mode (disables dropout) and track both losses. We cap val evaluation to
a few batches to keep it fast.

In [ ]:
@torch.no_grad()
def evaluate(model, loader, max_batches=10):
    model.eval()
    losses = []
    for i, (xb, yb) in enumerate(loader):
        if i >= max_batches:
            break
        xb, yb = xb.to(device), yb.to(device)
        _, loss = model(xb, yb)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))

history = {'train': [], 'val': [], 'train_ppl': [], 'val_ppl': []}

for epoch in range(EPOCHS):
    model.train()
    epoch_losses = []
    for step, (xb, yb) in enumerate(train_loader):
        if step >= steps_per_epoch:
            break
        xb, yb = xb.to(device), yb.to(device)
        _, loss = model(xb, yb)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        epoch_losses.append(loss.item())

    train_loss = float(np.mean(epoch_losses))
    val_loss = evaluate(model, val_loader)
    history['train'].append(train_loss)
    history['val'].append(val_loss)
    history['train_ppl'].append(perplexity(train_loss))
    history['val_ppl'].append(perplexity(val_loss))
    print(f"epoch {epoch+1:2d}/{EPOCHS} | train {train_loss:.4f} (ppl {perplexity(train_loss):6.2f}) "
          f"| val {val_loss:.4f} (ppl {perplexity(val_loss):6.2f})")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ep = np.arange(1, EPOCHS + 1)
ax1.plot(ep, history['train'], 'o-', label='train')
ax1.plot(ep, history['val'], 's-', label='val')
ax1.axhline(math.log(vocab_size), color='gray', ls='--', alpha=0.6, label='random baseline')
ax1.set_xlabel('epoch'); ax1.set_ylabel('cross-entropy loss'); ax1.set_title('Loss'); ax1.legend()

ax2.plot(ep, history['train_ppl'], 'o-', label='train')
ax2.plot(ep, history['val_ppl'], 's-', label='val')
ax2.set_xlabel('epoch'); ax2.set_ylabel('perplexity'); ax2.set_title('Perplexity'); ax2.legend()
plt.tight_layout(); plt.show()

### Reading the curves

- **Both still falling, close together** → underfitting / more capacity or training to go.
- **Train ↓ while val flattens or rises** → overfitting; the gap is the generalization error.
- **Both low and flat** → about as good as this model/data allows.

On this 11 KB corpus the train loss will keep dropping while the val loss bottoms out and
then drifts up — the model is starting to memorize Shakespeare rather than learn it.

## Step 8: Overfitting on a Tiny Dataset

With ~800K parameters and only ~10K training characters, the model has far more capacity
than the data warrants, so it eventually memorizes the train split. You can see this as a
widening train/val gap in the plot above (the val curve is the one to trust).

Levers that fight overfitting — all already available here:

- **Dropout** — the model uses `dropout=0.1` (active in `.train()`, off in `.eval()`).
- **Weight decay** — we set `0.1` in AdamW.
- **Early stopping** — keep the checkpoint at the *lowest val loss*, not the last epoch.
- **More data** — the real fix; tiny corpora overfit no matter what.

In [ ]:
best_epoch = int(np.argmin(history['val'])) + 1
print(f"Lowest VAL loss at epoch {best_epoch}: {min(history['val']):.4f}")
print(f"Final train/val gap: {history['train'][-1]:.4f} / {history['val'][-1]:.4f} "
      f"(gap = {history['val'][-1] - history['train'][-1]:+.4f})")
print("Early stopping would keep the epoch with the lowest val loss above.")

## Step 9: Sanity Check — Overfit a Single Batch

The single most useful debugging technique in deep learning: take **one** batch and train
on it repeatedly. A correctly wired model + loss + optimizer should drive the loss to
~0 (it can trivially memorize a handful of sequences). If it *can't*, you have a bug —
a detached graph, a shape error, a frozen parameter, a broken target shift — and there's
no point launching a real run until this passes.

In [ ]:
sanity_model = make_model()
sanity_opt = torch.optim.AdamW(sanity_model.parameters(), lr=3e-3)

xb, yb = next(iter(train_loader))
xb, yb = xb.to(device), yb.to(device)

sanity_losses = []
sanity_model.train()
for step in range(200):
    _, loss = sanity_model(xb, yb)
    sanity_opt.zero_grad()
    loss.backward()
    sanity_opt.step()
    sanity_losses.append(loss.item())
    if step % 40 == 0 or step == 199:
        print(f"step {step:3d}: loss {loss.item():.5f}")

plt.figure(figsize=(7, 3))
plt.plot(sanity_losses)
plt.xlabel('step'); plt.ylabel('loss'); plt.title('Overfit a single batch (should -> ~0)')
plt.yscale('log'); plt.tight_layout(); plt.show()
print(f"\nFinal single-batch loss: {sanity_losses[-1]:.5f}  -> wiring is correct.")
del sanity_model

## Step 10: Scaling Techniques

The loop above works on a laptop. Scaling to real models needs a few standard tricks.
These snippets are illustrative — they show the *pattern*; we run only the cheap parts.

### Gradient accumulation — bigger effective batch than fits in memory

Run several micro-batches, summing their (scaled) gradients, and step once. The effective
batch size becomes `micro_batch * accum_steps` without the memory cost of one big batch.

In [ ]:
# Pattern: effective batch = BATCH_SIZE * accum_steps, stepping once per group.
accum_steps = 4
accum_model = make_model()
accum_opt = torch.optim.AdamW(accum_model.parameters(), lr=3e-3)
accum_model.train()

batch_iter = iter(train_loader)
accum_opt.zero_grad()
for micro in range(accum_steps):
    xb, yb = next(batch_iter)
    xb, yb = xb.to(device), yb.to(device)
    _, loss = accum_model(xb, yb)
    (loss / accum_steps).backward()   # scale so summed grads = mean over big batch
print(f"Accumulated grads over {accum_steps} micro-batches "
      f"(effective batch = {BATCH_SIZE * accum_steps}).")
torch.nn.utils.clip_grad_norm_(accum_model.parameters(), 1.0)
accum_opt.step()                       # single optimizer step for the whole group
accum_opt.zero_grad()
print("Took one optimizer step for the effective batch.")
del accum_model

### Mixed precision — faster, less memory (GPU)

`torch.autocast` runs ops in bf16/fp16 where it's safe; `GradScaler` rescales the loss to
keep fp16 gradients from underflowing. This is a big speed/memory win on GPU and a no-op
benefit on CPU, so we **guard** on device.

In [ ]:
# CUDA pattern (guarded — executes the no-scaler path on CPU).
use_amp = (device == 'cuda')
amp_model = make_model()
amp_opt = torch.optim.AdamW(amp_model.parameters(), lr=3e-3)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

xb, yb = next(iter(train_loader))
xb, yb = xb.to(device), yb.to(device)

amp_opt.zero_grad()
with torch.autocast(device_type=('cuda' if use_amp else 'cpu'), dtype=torch.bfloat16, enabled=use_amp):
    _, loss = amp_model(xb, yb)
scaler.scale(loss).backward()
scaler.unscale_(amp_opt)                      # unscale before clipping
torch.nn.utils.clip_grad_norm_(amp_model.parameters(), 1.0)
scaler.step(amp_opt)
scaler.update()
print(f"AMP step done (use_amp={use_amp}), loss={loss.item():.4f}")
del amp_model

### Checkpointing — save and resume

Save model weights, optimizer state, and bookkeeping so a run can resume (or so you can
keep the best-val checkpoint). Save *everything* needed to continue, not just the weights.

In [ ]:
import os, tempfile

ckpt_path = os.path.join(tempfile.gettempdir(), 'gpt_ckpt.pt')
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'epoch': EPOCHS,
    'vocab_size': vocab_size,
    'seq_len': SEQ_LEN,
}, ckpt_path)
print(f"Saved checkpoint -> {ckpt_path} ({os.path.getsize(ckpt_path)/1024:.1f} KB)")

# Resume into a fresh model.
reloaded = make_model()
ckpt = torch.load(ckpt_path, map_location=device)
reloaded.load_state_dict(ckpt['model_state_dict'])
val_reloaded = evaluate(reloaded, val_loader, max_batches=5)
print(f"Reloaded model val loss: {val_reloaded:.4f} (matches trained model)")
del reloaded

In [ ]:
# Quick qualitative check: the trained model generates Shakespeare-ish text.
model.eval()
prompt = "The "
ids = torch.tensor([tokenizer.encode(prompt)], device=device)
out = model.generate(ids, max_new_tokens=120, temperature=0.8, top_k=20)
print(tokenizer.decode(out[0].tolist()))

## Step 11: The Bigger Picture

Training a GPT on Shakespeare is exactly the **pretraining** objective — next-token
prediction over raw text — just at a toy scale. Production LLMs stack additional stages
on top of it:

| Stage | Objective | Data | What it produces |
|---|---|---|---|
| **Pretraining** | Next-token prediction (what we did) | Trillions of tokens of raw web/text | A base model with broad knowledge & fluency |
| **Fine-tuning** | Same next-token loss, narrower data | Domain/task-specific corpus | A model specialized to a domain |
| **Instruction tuning (SFT)** | Next-token loss on (instruction, response) pairs | Curated demonstrations | A model that *follows instructions* |
| **RLHF / RLAIF** | Optimize a reward model via PPO/DPO | Human (or AI) preference comparisons | A model aligned to preferences (helpful, harmless) |

Key point: the **core mechanism never changes** — it's the same transformer minimizing the
same cross-entropy, with gradient clipping, AdamW, warmup+cosine LR, and the scaling tricks
above. What changes across stages is the *data* and the *objective framing*. Everything you
built in this notebook is the foundation the entire stack rests on.

> **Built from scratch later in this repo:** the **SFT** and **RLHF/DPO** rows above are no
> longer just names. **`12_instruction_tuning_and_lora.ipynb`** implements instruction
> tuning (with loss masking) and **LoRA**; **`13_preference_alignment.ipynb`** implements a
> reward model, RLHF (conceptually), and **DPO** — each on toy data, exactly the way this
> notebook implements pretraining.

---

## Summary

We turned the GPT from notebooks 02–07 into a *trained* model and learned the machinery
behind it.

### Key Takeaways

- **Objective:** training is maximum likelihood = minimizing cross-entropy on next-token
  prediction; logits `(B,T,V)` and targets `(B,T)` are flattened to `(B*T, V)` / `(B*T,)`.
- **Perplexity** = `exp(loss)`; the random baseline is `~vocab_size`. It's the
  interpretable lens on the loss.
- **Train/val split** with a shared tokenizer; the **val curve** is what tells you about
  generalization.
- **AdamW** is standard for transformers because it *decouples* weight decay from the
  adaptive gradient step.
- **Warmup + cosine decay** stabilizes early training (Adam's moment estimates are unreliable
  at the start) and anneals cleanly to the end.
- **Gradient clipping** (`clip_grad_norm_`, max 1.0) caps the occasional gradient spike.
- A correct loop: `zero_grad → forward → backward → clip → step → scheduler.step`, with a
  no-grad `eval()` pass for val loss.
- On a tiny corpus the model **overfits**; fight it with dropout, weight decay, and
  early stopping — but more data is the real fix.
- **Overfit-a-single-batch** is the first debugging step: if loss won't go to ~0, your
  wiring is broken.
- **Scaling:** gradient accumulation (bigger effective batch), mixed precision
  (`autocast` + `GradScaler` on GPU), and checkpointing for save/resume.
- The same machinery underlies real LLMs; **pretraining → fine-tuning → instruction tuning
  → RLHF** differ in data and framing, not in the core training mechanism.

### What's next

In **`09_inference.ipynb`** we take a trained model and focus on getting good *output* from
it: decoding strategies (greedy, temperature, top-k, top-p), KV caching, and the
speed/quality tradeoffs of inference.